# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset on second primary colorectal cancer using the `mlcroissant` library. All references to data structures such as record sets, fields, and columns are made via their Croissant `@id`s, ensuring consistency with the dataset schema.

### Dataset Source
This dataset is described using a Croissant schema. The schema and associated data files are referenced via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install the mlcroissant library if not already present
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`. The dataset object will give programmatic access to metadata and record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do not treat like dict or list

print(f"Dataset loaded: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review the available record sets and their fields by Croissant `@id`s.

This helps determine which tables and variables are available for loading and analysis.

In [ ]:
# List all record set @id's and show their fields by referencing @id

print("Available record sets (by `@id`):")
for obj in dataset.schema['@graph']:
    if obj.get('@type') == 'cr:RecordSet':
        record_set_id = obj['@id']
        print(f"  RecordSet: {record_set_id}")
        fields = obj.get('cr:field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            if isinstance(field, dict) and '@id' in field:
                print(f"    Field: {field['@id']}")
            elif isinstance(field, str):
                print(f"    Field: {field}")
        print()
# Store record set ids for later use
record_set_ids = [obj['@id'] for obj in dataset.schema['@graph'] if obj.get('@type') == 'cr:RecordSet']
if not record_set_ids:
    print('No record sets found in schema.')

## 3. Data Extraction
In this section, we load one or more record sets into Pandas DataFrames. All references use the full `@id` as previously printed.

We extract each record set's records into a dictionary keyed by the record set `@id`.

In [ ]:
dataframes = {}
available_record_sets = record_set_ids  # From previous code block

for record_set in available_record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        if records:
            dataframes[record_set] = pd.DataFrame(records)
            print(f"Loaded record set: {record_set}, shape: {dataframes[record_set].shape}")
            print(f"  Fields (@id): {list(dataframes[record_set].columns)}\n")
        else:
            print(f"Record set {record_set} returned 0 records.")
    except Exception as e:
        print(f"Error loading {record_set}: {e}")

# If at least one DataFrame loaded, show a preview
if dataframes:
    example_record_set = list(dataframes.keys())[0]
    print(f"\nPreview of first 5 rows from record set: {example_record_set}")
    display(dataframes[example_record_set].head())
else:
    print('No dataframes loaded. Cannot continue with data exploration.')

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic EDA: filter records, normalize values, and group by categorical fields.
Below, you'll need to supply the actual `@id` of a numeric field present in your selected record set. You can adapt these operations for any available field.

In [ ]:
# --- Adapting for available field @ids ---
# For demonstration, get the first loaded dataframe and show its columns (@id's)
if dataframes:
    record_set_id = example_record_set
    df = dataframes[record_set_id]
    print(f"Fields available in record set {record_set_id}:")
    print(df.columns.tolist())

    # Select a numeric field (adapt this line with the correct @id as needed)
    # If unsure, pick first integer/float column
    numeric_field = None
    for c in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field = c
                break
        except Exception:
            continue
    if numeric_field is None:
        print('No numeric field found for analysis. Please update the field selection to match your data.')
    else:
        print(f"Selected numeric field: {numeric_field}")

        # Example: Filter records above the median
        threshold = df[numeric_field].median() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field if available
        group_field = None
        for c in df.columns:
            if c == numeric_field:
                continue
            if pd.api.types.is_object_dtype(df[c]) and len(df[c].unique()) < 10:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            display(grouped_df)
        else:
            print('No categorical field suitable for grouping was found.')
else:
    print('No data loaded to perform EDA.')

## 5. Visualization
Below are example visualizations using matplotlib and seaborn to understand distributions and categorical relationships.

All variable references are via `@id` only.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    # Histogram
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a grouping field was found in previous cell
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to programmatically access and explore a FAIR dataset described by the Croissant schema using `mlcroissant`. All dataset structures (record sets, fields, columns) were referenced by their `@id`. 

- We loaded the dataset and listed available record sets and field `@id`s.
- We showed how to extract a record set into a DataFrame for inspection.
- We performed simple filtering, normalization, and grouping based entirely on Croissant entity `@id`s.
- We visualized distributions using standard plotting libraries.

This interoperable approach accommodates future datasets with minimum refactoring, as all logic is driven by standard `@id` references.